# Exercise 4-03: Design a Controlled LLM Experiment & Detect Data Leakage (Optional Worksheet)

**Chapter 4: Experimentation, Fine-Tuning, and Model Evaluation**  
**Lab Type:** Interactive Worksheet Notebook | **Estimated Time:** 30–40 minutes  
**Exam Domains Covered:** Experimentation (22%), Data Analysis & Visualization (14%)

---

## Learning Objectives
By the end of this interactive worksheet, you will be able to:
1. Formulate a **falsifiable hypothesis ($H_1$)** and **null hypothesis ($H_0$)** for an LLM or RAG engineering experiment.
2. Correctly classify **Independent**, **Dependent**, **Controlled**, and **Confounding** variables in scenario-based exam questions.
3. Select the proper **Data Split Strategy** (**Random**, **Stratified**, **Time-Based / Temporal**, or **Group-Based** by user/conversation/document ID) to prevent **Data Leakage**.
4. Run an automated **Train-Test Leakage & $n$-gram Benchmark Contamination Detector** in Python.

---
## 1. Conceptual Foundation: The Scientific Method in AI Experimentation

In **Chapter 4 (*Experimental Design and the Scientific Method*)**, the book outlines the anatomy of a valid, controlled AI experiment:

| Experimental Element | Definition | LLM / RAG Example |
| :--- | :--- | :--- |
| **Falsifiable Hypothesis ($H_1$)** | Specific, measurable prediction linking ONE change to an expected outcome | *"Adding `nv-rerankqa-mistral-4b-v3` reranking (top-10 $\rightarrow$ top-4) will increase RAG answer faithfulness by $\ge 8\%$ while adding $< 150\text{ ms}$ P95 latency."* |
| **Null Hypothesis ($H_0$)** | Default assumption that the intervention has **no real effect** (any observed difference is random noise) | *"Reranking produces no statistically significant change in answer faithfulness."* |
| **Independent Variable** | The **single factor** intentionally varied by the experimenter | `use_reranker` (`False` vs. `True`) or `lora_rank` (`4, 8, 16`) |
| **Dependent Variable(s)** | The outcome metric(s) measured to evaluate the effect | Answer Faithfulness Score, ROUGE-L, P95 Latency (ms) |
| **Controlled Variables** | Factors explicitly **held constant** across all runs | Base LLM (`llama-3.1-8b`), `temperature=0.1`, embedding model, evaluation test set, `seed=42` |
| **Confounding Variable** | An **uncontrolled factor that changes simultaneously** with the independent variable, making it impossible to know which change caused the result! | Switching the base model from 8B to 70B *at the same time* as changing the prompt template |

> **EXAM TIP (Chapter 4):** Many NCA-GENL questions present a flawed experiment where an engineer changes **two or three variables at once** (e.g., increasing LoRA rank $r$, doubling the dataset size, and switching the optimizer) and asks why the experiment is invalid. The answer is **confounding variables** (violating the **One-Variable-at-a-Time / Ablation** principle)!

In [1]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import hashlib
import re
import pandas as pd
import numpy as np

# Interactive Worksheet Part 1: Controlled Experiment Validator
EXPERIMENT_SCENARIOS = [
    {
        "scenario_id": "EXP-01",
        "description": (
            "Team wants to test if 4-bit AWQ quantization degrades medical QA accuracy by less than 1.5% "
            "while cutting VRAM usage in half on Llama-3-8B."
        ),
        "hypothesis": "Quantizing Llama-3-8B from FP16 to 4-bit AWQ reduces GPU VRAM by >=45% while keeping PubMedQA accuracy within 1.5% of the FP16 baseline.",
        "independent_var": "Weight Precision (FP16 vs. 4-bit AWQ)",
        "dependent_vars": ["PubMedQA Accuracy (%)", "Peak GPU VRAM (GB)", "Mean ITL (ms)"],
        "controlled_vars": ["Model architecture (Llama-3-8B)", "Prompt template (0-shot)", "Temperature (0.0)", "Evaluation split (500 held-out questions)"],
        "flawed_confounder_example": "Evaluating FP16 with 0-shot prompts but evaluating 4-bit AWQ with 5-shot Chain-of-Thought prompts!",
    },
    {
        "scenario_id": "EXP-02",
        "description": (
            "Team wants to test if increasing chunk_overlap from 0 to 80 characters improves RAG recall@5 "
            "on legal contracts."
        ),
        "hypothesis": "Setting chunk_overlap=80 (at fixed chunk_size=500) increases Retrieval Recall@5 by >=6% on the 200-query legal benchmark compared to chunk_overlap=0.",
        "independent_var": "chunk_overlap (0 vs. 80 characters)",
        "dependent_vars": ["Retrieval Recall@5", "Index Chunk Count", "Ingestion Time (s)"],
        "controlled_vars": ["chunk_size=500", "Embedding model (nv-embedqa-e5-v5)", "Vector index (FAISS FlatIP)", "200-query test set"],
        "flawed_confounder_example": "Changing chunk_overlap from 0 to 80 while simultaneously switching embeddings from Word2Vec to nv-embedqa-e5-v5!",
    },
]

for sc in EXPERIMENT_SCENARIOS:
    print("=" * 85)
    print(f"[{sc['scenario_id']}] {sc['description']}")
    print("-" * 85)
    print(f"  Falsifiable Hypothesis : {sc['hypothesis']}")
    print(f"  Independent Variable   : {sc['independent_var']}")
    print(f"  Dependent Variables    : {', '.join(sc['dependent_vars'])}")
    print(f"  Controlled Variables   : {', '.join(sc['controlled_vars'])}")
    print(f"  Confounder to Avoid    : {sc['flawed_confounder_example']}\n")

[EXP-01] Team wants to test if 4-bit AWQ quantization degrades medical QA accuracy by less than 1.5% while cutting VRAM usage in half on Llama-3-8B.
-------------------------------------------------------------------------------------
  Falsifiable Hypothesis : Quantizing Llama-3-8B from FP16 to 4-bit AWQ reduces GPU VRAM by >=45% while keeping PubMedQA accuracy within 1.5% of the FP16 baseline.
  Independent Variable   : Weight Precision (FP16 vs. 4-bit AWQ)
  Dependent Variables    : PubMedQA Accuracy (%), Peak GPU VRAM (GB), Mean ITL (ms)
  Controlled Variables   : Model architecture (Llama-3-8B), Prompt template (0-shot), Temperature (0.0), Evaluation split (500 held-out questions)
  Confounder to Avoid    : Evaluating FP16 with 0-shot prompts but evaluating 4-bit AWQ with 5-shot Chain-of-Thought prompts!

[EXP-02] Team wants to test if increasing chunk_overlap from 0 to 80 characters improves RAG recall@5 on legal contracts.
--------------------------------------------------------

---
## 2. Selecting the Right Train / Validation / Test Split Strategy

In **Chapter 4 (*Data Splitting Strategies and Preventing Data Leakage*)**, choosing the wrong split strategy causes **Data Leakage**, which inflates validation/test metrics in the lab and causes catastrophic failure in production:

| Dataset Structure | Correct Split Strategy | Why Simple Random Splitting Fails (Leakage Risk!) |
| :--- | :--- | :--- |
| **Imbalanced Classification** (e.g., 97% benign prompts, 3% jailbreak attacks) | **Stratified Split** | A small random validation split might contain almost zero minority-class examples |
| **Time-Series / Evolving Logs** (e.g., support tickets or news from Jan–Dec) | **Time-Based (Temporal) Split** (Train: Jan–Sep, Val: Oct, Test: Nov–Dec) | Random splitting lets the model train on *future* November events to predict *past* October tickets (**Temporal Leakage / Look-Ahead Bias**) |
| **Multiple Samples per Entity** (e.g., 10 turns from the same user conversation, or 20 chunks from the same PDF document) | **Group-Based Split (`GroupShuffleSplit`)** grouped by `user_id`, `conversation_id`, or `doc_id` | Random splitting puts 8 chunks of a PDF in Train and 2 chunks of the *same PDF* in Test, measuring memorization rather than generalization! |

In [2]:
# Demonstrate why Group-Based Splitting (by doc_id) prevents leakage vs. Naive Random Splitting
raw_chunks_df = pd.DataFrame([
    {"chunk_id": f"DOC_{doc_num}#c{c_num}", "doc_id": f"DOC_{doc_num}", "patient_id": f"PAT_{doc_num}",
     "text": f"Patient PAT_{doc_num} clinical note section {c_num}: rare genomic marker XZ-{doc_num}99 detected."}
    for doc_num in range(1, 6)
    for c_num in range(1, 5)
])

rng = np.random.default_rng(42)
# 1. Naive Random Split (80% train, 20% test)
shuffled_idx = rng.permutation(len(raw_chunks_df))
naive_train = raw_chunks_df.iloc[shuffled_idx[:16]]
naive_test = raw_chunks_df.iloc[shuffled_idx[16:]]
leaked_docs_naive = set(naive_train["doc_id"]) & set(naive_test["doc_id"])

# 2. Group-Based Split by doc_id (Docs 1..4 in Train, Doc 5 in Test)
group_train = raw_chunks_df[raw_chunks_df["doc_id"].isin(["DOC_1", "DOC_2", "DOC_3", "DOC_4"])]
group_test = raw_chunks_df[raw_chunks_df["doc_id"].isin(["DOC_5"])]
leaked_docs_group = set(group_train["doc_id"]) & set(group_test["doc_id"])

print("=== SPLIT STRATEGY COMPARISON ON MULTI-CHUNK DOCUMENTS ===")
print(f"1. Naive Random Split -> Overlapping doc_ids in BOTH Train & Test : {sorted(leaked_docs_naive)} (SEVERE LEAKAGE!)")
print(f"2. Group-Based Split  -> Overlapping doc_ids in BOTH Train & Test : {sorted(leaked_docs_group)} (ZERO LEAKAGE!)")

=== SPLIT STRATEGY COMPARISON ON MULTI-CHUNK DOCUMENTS ===
1. Naive Random Split -> Overlapping doc_ids in BOTH Train & Test : ['DOC_1', 'DOC_3', 'DOC_4', 'DOC_5'] (SEVERE LEAKAGE!)
2. Group-Based Split  -> Overlapping doc_ids in BOTH Train & Test : [] (ZERO LEAKAGE!)


---
## 3. Automated Data Leakage & $n$-Gram Benchmark Contamination Detector

For Large Language Models, a particularly insidious form of data leakage is **Benchmark Contamination**: test questions from evaluation benchmarks (like MMLU, GSM8K, or your company's golden test set) accidentally appear verbatim or near-verbatim inside the pretraining or fine-tuning corpus!

Let's write a Python `detect_corpus_contamination()` utility that checks for:
1. **Exact Normalized String / SHA-256 Hash Overlap** between `train_texts` and `eval_texts`.
2. **High-Order $n$-Gram Overlap (e.g., 8-gram or 10-gram contamination)**—the exact technique used in GPT-4, Llama-3, and NeMo Curator technical reports to strip contaminated benchmark items before scoring!

In [3]:
def normalize_for_dedup(text: str) -> str:
    return " ".join(re.findall(r"[a-z0-9]+", text.lower()))


def extract_word_ngrams(text: str, n: int = 8) -> set:
    words = normalize_for_dedup(text).split()
    if len(words) < n:
        return {tuple(words)} if words else set()
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


def detect_corpus_contamination(train_corpus: list, eval_benchmark: list, ngram_size: int = 8) -> pd.DataFrame:
    train_hashes = {hashlib.sha256(normalize_for_dedup(t).encode()).hexdigest(): i for i, t in enumerate(train_corpus)}
    train_ngrams = set()
    for t in train_corpus:
        train_ngrams.update(extract_word_ngrams(t, n=ngram_size))

    report = []
    for idx, eval_item in enumerate(eval_benchmark):
        norm_eval = normalize_for_dedup(eval_item)
        eval_hash = hashlib.sha256(norm_eval.encode()).hexdigest()
        exact_match = eval_hash in train_hashes

        eval_ngs = extract_word_ngrams(eval_item, n=ngram_size)
        contaminated_ngs = eval_ngs & train_ngrams
        overlap_ratio = len(contaminated_ngs) / max(1, len(eval_ngs))

        status = "CLEAN"
        if exact_match:
            status = "EXACT_LEAKAGE"
        elif overlap_ratio >= 0.5:
            status = f"NEAR_DUPLICATE_{ngram_size}GRAM_LEAK"

        report.append({
            "Eval_ID": f"TEST-{idx+1:02d}",
            "Status": status,
            "Exact_Hash_Match": exact_match,
            f"{ngram_size}Gram_Overlap_%": round(overlap_ratio * 100, 1),
            "Eval_Question_Preview": eval_item[:68] + "...",
        })
    return pd.DataFrame(report)


sample_train_corpus = [
    "To launch an NVIDIA NIM container on port 8000, run docker run --gpus all -e NGC_API_KEY.",
    "Which numerical precision format preserves the 8-bit dynamic exponent range of FP32 while using only 16 bits per parameter? Answer: BF16 (BFloat16).",
    "In a transformer model, scaled dot-product attention divides QK^T by the square root of the head dimension d_k to prevent vanishing gradients in softmax.",
]

sample_eval_benchmark = [
    # Item 1: Clean unseen test question
    "How does NVIDIA NeMo Guardrails use Colang flows to enforce topical boundaries in customer chatbots?",
    # Item 2: Exact copy leaked into train_corpus!
    "Which numerical precision format preserves the 8-bit dynamic exponent range of FP32 while using only 16 bits per parameter? Answer: BF16 (BFloat16).",
    # Item 3: Paraphrased prefix, but shares a verbatim 10-word span with train_corpus[2]!
    "Explain why scaled dot-product attention divides QK^T by the square root of the head dimension d_k to prevent saturation.",
]

df_leak = detect_corpus_contamination(sample_train_corpus, sample_eval_benchmark, ngram_size=8)
print("=== AUTOMATED TRAIN-TEST CONTAMINATION AUDIT REPORT ===")
print(df_leak.to_string(index=False))

=== AUTOMATED TRAIN-TEST CONTAMINATION AUDIT REPORT ===
Eval_ID                    Status  Exact_Hash_Match  8Gram_Overlap_%                                                   Eval_Question_Preview
TEST-01                     CLEAN             False              0.0 How does NVIDIA NeMo Guardrails use Colang flows to enforce topical ...
TEST-02             EXACT_LEAKAGE              True            100.0 Which numerical precision format preserves the 8-bit dynamic exponen...
TEST-03 NEAR_DUPLICATE_8GRAM_LEAK             False             80.0 Explain why scaled dot-product attention divides QK^T by the square ...


---
## 4. Hands-On Exercise: Diagnose the Experimental Flaw

Read each of the 3 scenarios in the code cell below and match it to its primary methodological flaw (`"CONFOUNDING_VARIABLES"`, `"TEMPORAL_DATA_LEAKAGE"`, or `"PREPROCESSING_FIT_ON_FULL_DATASET"`).

In [4]:
# Exercise & Solution: Experimental Flaw Diagnostic Quiz
quiz_cases = [
    {
        "case": "Engineer standardizes features (mean/std) and builds TF-IDF vocabulary on the entire 10,000-row dataset BEFORE splitting into Train and Test.",
        "correct_flaw": "PREPROCESSING_FIT_ON_FULL_DATASET",
        "remedy": "Split into Train/Val/Test FIRST, then call .fit_transform() on Train and .transform() on Val/Test.",
    },
    {
        "case": "Engineer switches from Llama-3-8B to Llama-3-70B AND simultaneously adds a 5-shot Chain-of-Thought prompt, then concludes CoT caused the +18% accuracy jump.",
        "correct_flaw": "CONFOUNDING_VARIABLES",
        "remedy": "Run an ablation study holding the model constant (8B Zero-Shot vs. 8B CoT) to isolate the effect of CoT.",
    },
    {
        "case": "Engineer randomly shuffles 2023-2025 financial earnings calls into an 80/20 train/test split, allowing the model to see Q4 2025 macro outcomes while predicting Q1 2024 risk.",
        "correct_flaw": "TEMPORAL_DATA_LEAKAGE",
        "remedy": "Use a chronological Time-Based Split where all Test timestamps occur strictly after all Train/Val timestamps.",
    },
]

for idx, q in enumerate(quiz_cases, 1):
    print(f"Case {idx}: {q['case']}")
    print(f"  -> Primary Flaw : {q['correct_flaw']}")
    print(f"  -> Correct Fix  : {q['remedy']}\n")

Case 1: Engineer standardizes features (mean/std) and builds TF-IDF vocabulary on the entire 10,000-row dataset BEFORE splitting into Train and Test.
  -> Primary Flaw : PREPROCESSING_FIT_ON_FULL_DATASET
  -> Correct Fix  : Split into Train/Val/Test FIRST, then call .fit_transform() on Train and .transform() on Val/Test.

Case 2: Engineer switches from Llama-3-8B to Llama-3-70B AND simultaneously adds a 5-shot Chain-of-Thought prompt, then concludes CoT caused the +18% accuracy jump.
  -> Primary Flaw : CONFOUNDING_VARIABLES
  -> Correct Fix  : Run an ablation study holding the model constant (8B Zero-Shot vs. 8B CoT) to isolate the effect of CoT.

Case 3: Engineer randomly shuffles 2023-2025 financial earnings calls into an 80/20 train/test split, allowing the model to see Q4 2025 macro outcomes while predicting Q1 2024 risk.
  -> Primary Flaw : TEMPORAL_DATA_LEAKAGE
  -> Correct Fix  : Use a chronological Time-Based Split where all Test timestamps occur strictly after all Train/Val t